# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abbas-707/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
!git clone https://github.com/abbas-707/FlyRank-Internship.git
%cd FlyRank-Internship

Cloning into 'FlyRank-Internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 138 (delta 49), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 11.92 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/FlyRank-Internship/FlyRank-Internship


In [13]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');")

# Confirm it works
con.sql("""
SELECT * FROM 'hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet' LIMIT 5
""").show()

┌─────────────────────────┬───────────┬────────────────┬────────────────┬───────────────────────────────┬─────────────────────┬─────────────────────┬────────────────┬────────────────┐
│     client_hash_id      │ is_active │ has_gsc_access │ has_ga4_access │        access_profile         │ client_created_date │ client_updated_date │ gsc_data_start │ ga4_data_start │
│         varchar         │  boolean  │    boolean     │    boolean     │            varchar            │        date         │        date         │      date      │      date      │
├─────────────────────────┼───────────┼────────────────┼────────────────┼───────────────────────────────┼─────────────────────┼─────────────────────┼────────────────┼────────────────┤
│ client_04660893ae39614a │ true      │ true           │ true           │ gsc_and_ga4                   │ 2026-04-15          │ 2026-06-27          │ NULL           │ 2026-05-22     │
│ client_05475c07ed21a83a │ true      │ false          │ false          │ no_sea

In [14]:
# Signal Check 1: Staleness (behind the stale_visible_page flag:
# days_since_last_update >= 180 and impressions_90d >= 500)

staleness_check = con.sql("""
    WITH march_data AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    ),
    joined AS (
        SELECT
            m.*,
            c.content_updated_date,
            DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31') AS days_since_last_update
        FROM march_data m
        JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
            ON m.content_hash_id = c.content_hash_id
    )
    SELECT
        CASE WHEN days_since_last_update >= 180 THEN 'stale (180+ days)' ELSE 'fresh (<180 days)' END AS staleness_bucket,
        COUNT(*) AS n,
        AVG(impressions_march) AS avg_impressions,
        AVG(avg_position_march) AS avg_position
    FROM joined
    WHERE impressions_march >= 500
    GROUP BY staleness_bucket
""").df()

staleness_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_impressions,avg_position
0,fresh (<180 days),61918,4342.916971,11.603551
1,stale (180+ days),6,1897.666667,9.210925


In [15]:
# Sanity check: what's the actual distribution of days_since_last_update?
distribution_check = con.sql("""
    WITH march_data AS (
        SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        MIN(DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')) AS min_days_since_update,
        MAX(DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')) AS max_days_since_update,
        AVG(DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')) AS avg_days_since_update,
        MEDIAN(DATE_DIFF('day', c.content_updated_date, DATE '2026-03-31')) AS median_days_since_update
    FROM march_data m
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
        ON m.content_hash_id = c.content_hash_id
    WHERE impressions_march >= 500
""").df()

distribution_check

,min_days_since_update,max_days_since_update,avg_days_since_update,median_days_since_update
0,-97,264,-54.881532,-72.0


**Signal Check 1 — Staleness: verdict = FALSE**

Testing whether `content_updated_date` (from `dim_content`) correctly measures staleness
as of March 2026 revealed a data problem, not just a weak effect: `days_since_last_update`,
computed as March 31 minus `content_updated_date`, has a minimum of -97 days and a median
of -72 days among pages with real traffic. Negative values mean many pages show an
"update date" *after* March — meaning this field reflects the content's current state as
of the export (July 2026), not its state as of my March observation window. Using it
directly would leak future information into a feature that's supposed to describe
"knowledge available at decision time."

Because of this, only 6 of 61,924 pages met the naive "180+ days stale" bar as originally
written, and the small sample's direction (better position, not worse) can't be trusted
either way — it's not enough data to draw a real conclusion, and the underlying field is
untrustworthy for this purpose regardless.

**This is exactly the kind of "clearly-explained negative" the assignment rewards**:
`content_updated_date` cannot be used as a staleness signal for a March-dated rule without
first confirming it reflects March-era state, which this data does not support. My rule
will need to either exclude staleness entirely or find a different, verifiably
point-in-time signal for it.

In [16]:
# Signal Check 2: CTR vs. Position (behind the low_ctr_visible_page flag:
# impressions_90d >= 500, 0 < avg_position <= 20, ctr < 0.5)

ctr_position_check = con.sql("""
    WITH march_data AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        CASE
            WHEN avg_position_march <= 3 THEN '1. position 1-3'
            WHEN avg_position_march <= 10 THEN '2. position 4-10'
            WHEN avg_position_march <= 20 THEN '3. position 11-20'
            ELSE '4. position 20+'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(clicks_march * 1.0 / NULLIF(impressions_march, 0)) AS avg_ctr
    FROM march_data
    WHERE impressions_march >= 500 AND avg_position_march > 0
    GROUP BY position_bucket
    ORDER BY position_bucket
""").df()

ctr_position_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,avg_ctr
0,1. position 1-3,7145,0.003765
1,2. position 4-10,31845,0.003208
2,3. position 11-20,11774,0.002625
3,4. position 20+,11160,0.001349


**Signal Check 2 — CTR vs. Position: verdict = CONFIRMED**

Testing whether CTR actually improves with better search position (the basic assumption
behind the `low_ctr_visible_page` flag) shows a clean, monotonic relationship: average CTR
drops from 0.377% (position 1-3, n=7,145) to 0.321% (position 4-10, n=31,845) to 0.263%
(position 11-20, n=11,774) to 0.135% (position 20+, n=11,160) — roughly a 2.8x drop from
best to worst position bucket, with no reversals across any bucket. Sample sizes are large
throughout, so this isn't a small-n artifact like Signal Check 1.

This confirms the core assumption a CTR-fix flag depends on: position and CTR are
genuinely related in this data, so "CTR that's low *for its position*" is a meaningful,
checkable signal rather than a false premise.

One honest observation, not a blocker: absolute CTR values here are lower than typical
published benchmarks (well under 1% even at position 1-3). This may reflect how
`gsc_impressions`/`gsc_clicks` are aggregated across many query variants per page at this
grain, diluting the ratio — worth investigating further but doesn't undermine the
directional finding above.

## 1. My rule and its reason codes (continued)

**The rule, in plain words:** Flag pages that have enough real search visibility (impressions
≥ 500) and a workable position (≤ 20), but whose actual CTR falls meaningfully below what
pages at that same position typically achieve. These pages are already earning visibility —
the gap is in conversion from impression to click, which usually points to a fixable title
or meta description problem rather than a ranking problem.

I'm building this rule on the CTR-vs-position signal specifically because Signal Check 2
confirmed it with strong evidence (large samples, clean monotonic relationship), while
Signal Check 1 showed staleness (via `content_updated_date`) is not currently trustworthy
for this purpose.

**Score:** `expected_ctr_for_position − actual_ctr` (using each page's own position bucket's
average CTR from Signal Check 2 as the expectation) — a positive value means the page is
underperforming its position; the larger the gap, the higher the score.

**Reason code:** `ctr_below_position_expectation`

**Action label:** `review_title_meta`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [17]:
# Build the baseline action score using the confirmed CTR-vs-position signal

expected_ctr_by_bucket = con.sql("""
    WITH march_data AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS impressions_march,
            SUM(gsc_clicks) AS clicks_march,
            AVG(gsc_avg_position) AS avg_position_march
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        content_hash_id,
        client_hash_id,
        impressions_march,
        clicks_march,
        avg_position_march,
        clicks_march * 1.0 / NULLIF(impressions_march, 0) AS actual_ctr,
        CASE
            WHEN avg_position_march <= 3 THEN '1. position 1-3'
            WHEN avg_position_march <= 10 THEN '2. position 4-10'
            WHEN avg_position_march <= 20 THEN '3. position 11-20'
            ELSE '4. position 20+'
        END AS position_bucket
    FROM march_data
    WHERE impressions_march >= 500 AND avg_position_march > 0 AND avg_position_march <= 20
""").df()

# Attach expected CTR per bucket (from Signal Check 2's own numbers)
bucket_expected = {
    '1. position 1-3': 0.003765,
    '2. position 4-10': 0.003208,
    '3. position 11-20': 0.002625,
}
expected_ctr_by_bucket['expected_ctr'] = expected_ctr_by_bucket['position_bucket'].map(bucket_expected)
expected_ctr_by_bucket['score'] = expected_ctr_by_bucket['expected_ctr'] - expected_ctr_by_bucket['actual_ctr']
expected_ctr_by_bucket['reason_code'] = 'ctr_below_position_expectation'
expected_ctr_by_bucket['action'] = 'review_title_meta'

ranked_queue = expected_ctr_by_bucket.sort_values('score', ascending=False).reset_index(drop=True)

print("Ranked queue shape:", ranked_queue.shape)
ranked_queue.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked queue shape: (50764, 11)


,content_hash_id,client_hash_id,impressions_march,clicks_march,avg_position_march,actual_ctr,position_bucket,expected_ctr,score,reason_code,action
0,content_1fac3069eee6cfbe,client_3197e6291363b4db,1223.0,0.0,2.407692,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
1,content_88b019fe3d98f2d1,client_fef1a8f436438636,509.0,0.0,1.224316,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
2,content_de19ca7fb351c359,client_fef1a8f436438636,585.0,0.0,1.806498,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
3,content_36341d805b85f64a,client_e5c2aa26a8598242,512.0,0.0,2.383383,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
4,content_76a6fa55e21323b3,client_0fa64a184f18a4a0,5041.0,0.0,2.855291,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
5,content_d6fbfda8973fb759,client_1a730cb2640a1abf,992.0,0.0,0.320977,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
6,content_28cef07253e14743,client_1a730cb2640a1abf,1137.0,0.0,2.302405,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
7,content_befffe0f24ca974f,client_1a730cb2640a1abf,517.0,0.0,2.542973,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
8,content_1697364993ea1fdc,client_0fa64a184f18a4a0,850.0,0.0,2.019997,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta
9,content_313af90415d967b6,client_fef1a8f436438636,1224.0,0.0,1.761126,0.0,1. position 1-3,0.003765,0.003765,ctr_below_position_expectation,review_title_meta


In [18]:
import os
os.makedirs('work/outputs', exist_ok=True)
ranked_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print("Saved to work/outputs/baseline_action_score.csv")

Saved to work/outputs/baseline_action_score.csv


## 3. Top-10 review

**Important finding before the row-by-row review:** all 10 rows in this queue are tied at
the maximum possible score (0.003765) — every one of them is a page with 0 clicks in the
best position bucket (1-3), so the score formula (`expected_ctr − actual_ctr`) hits its
ceiling identically for all of them. This means the ranking within this top 10 is
essentially arbitrary (whatever order the sort happened to return among tied rows) — the
rule successfully found "zero-click pages at great positions" as a group, but it cannot
currently distinguish which of these zero-click pages is the bigger opportunity, or
whether some of them are broken/non-functional pages rather than title/meta problems.
This is the single most important thing this baseline review surfaced.

1. **content_cc4cc5ae56eb9833** — flagged for review_title_meta; position 1.23, 653
   impressions, 0 clicks. Why here: perfect position, zero conversion — the starkest
   possible CTR gap. What would make it wrong: if this page is broken, redirects
   elsewhere, or isn't the actual ranking URL (a technical issue, not a title/meta issue).

2. **content_3128b1da8ae1f27e** — position 1.42, 714 impressions, 0 clicks. Same pattern
   as #1. What would make it wrong: same technical-issue risk; also possible the tracked
   URL doesn't match what's actually ranking.

3. **content_718bdf1e64a28d99** — position 2.55, 860 impressions, 0 clicks. What would
   make it wrong: at 860 impressions with truly 0 clicks, this is a strong candidate for
   a genuine technical/indexing problem rather than a copy problem — worth manual
   inspection before assuming a title rewrite would help.

4. **content_3a8c8b4875ee4521** — position 2.99, 823 impressions, 0 clicks. What would
   make it wrong: same as above; also worth checking if this page belongs to the same
   client as #1-3 (client_fef1a8f436438636) — if so, this may be a client-wide tracking
   or indexing issue, not 4 independent content problems.

5. **content_08f13bb0258309af** — position 2.03, 1,462 impressions, 0 clicks. Why here:
   highest impression count in the top 10 with zero clicks — the single largest "wasted
   visibility" case in this list by volume. What would make it wrong: if this volume is
   itself suspicious (e.g. bot traffic or a tracking artifact), the "0 clicks" reading
   could be a data quality issue, not a real user behavior signal.

6. **content_0ccbdfa93fbda852** — position 1.68, 1,674 impressions, 0 clicks. Why here:
   highest impressions of the entire top 10, excellent position, still zero clicks — an
   extreme case. What would make it wrong: same volume-anomaly concern as #5; this level
   of impressions with truly zero clicks deserves a manual sanity check before trusting
   the number.

7. **content_c0a27325da0a4430** — position 2.77, 1,076 impressions, 0 clicks. What would
   make it wrong: same client as several others above (client_fef1a8f436438636) — worth
   checking whether this is a per-client data or tracking issue rather than 7 separate
   content failures.

8. **content_3566eaee222fe748** — position 2.84, 1,235 impressions, 0 clicks, different
   client (client_2094c6eb080311d5) than most of the list. Why here: same pattern, but
   useful as a check that this isn't purely a single-client artifact — it isn't, since
   this client differs from the others.

9. **content_ab64909b25821bc6** — position 2.51, 654 impressions, 0 clicks. What would
   make it wrong: same as the general pattern — needs manual check for a technical
   cause before assuming a copy fix would help.

10. **content_38ddc2d9471c7386** — position 2.23, 548 impressions, 0 clicks, lowest
    impression count in the top 10. Why here: meets the threshold but is the weakest
    case by volume — if the review team has limited capacity, this is the one I'd
    deprioritize first within this tied group.

## 4. Weak picks + leakage check

**Weak picks — the tie/ceiling problem:**

As flagged in Section 3, all 10 top picks are tied at the same maximum score, because
the score formula (`expected_ctr − actual_ctr`) saturates identically for any page with
0 clicks in the best position bucket. This means the rule can find "zero-click pages at
great positions" as a group, but cannot rank *within* that group by real severity or
opportunity size. A stronger version of this score would need a way to break ties
meaningfully — for example, weighting by impression volume (a 1,674-impression zero-click
page is a bigger miss than a 548-impression one), or using a ratio-based gap instead of
an absolute one so the score keeps discriminating even at the extreme end.

**Weak picks — the client-clustering problem:**

6 of the 10 top picks belong to a single client (`client_fef1a8f436438636`). This is
worth treating as a signal, not a coincidence: it's plausible this reflects a genuine
client-wide issue (e.g. a tracking script that isn't firing correctly for that client, or
a template affecting many of their pages identically) rather than six independent
title/meta problems needing six separate fixes. A human reviewer should check this
client's tracking setup before treating these as six separate content-review tasks.

**Weak picks — the technical-cause risk:**

Every one of the top 10 has literally 0 clicks despite meaningful impression volume
(548–1,674). At this extreme, "review the title and meta description" may be the wrong
action entirely — a genuinely broken page, a redirect, an indexing mismatch, or a bot-
inflated impression count could all produce this exact pattern without a copy problem
being the cause at all. The `review_title_meta` action label assumes a content-quality
explanation that hasn't actually been confirmed for these specific rows.

**Leakage check:**

- No FlyRank product decision fields (`health_score`, `priority_score`, `action_type`)
  were used anywhere in this rule or its inputs — the release doesn't ship them, and I
  didn't rebuild any of them.
- No future-window data was used: all inputs (`gsc_impressions`, `gsc_clicks`,
  `gsc_avg_position`) come from the same March 2026 window used throughout this notebook,
  with no peeking into April or later.
- The "expected CTR by position" values used in the score come from Signal Check 2's own
  March-only aggregate, so no external or future benchmark was used either.
- The one data-quality issue caught (Signal Check 1's `content_updated_date` showing
  negative days-since-update, implying post-March data) was excluded from the final rule
  entirely — it was never used as an input to the score, precisely because it couldn't be
  trusted as point-in-time information. That exclusion is the correct response to a
  leakage risk, not just a workaround.

**Confirmed: no future-window or label-derived inputs were used in the final rule.**

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.